In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("serkantysz/550k-spotify-songs-audio-lyrics-and-genres")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import mutual_info_classif # MI
from sklearn.preprocessing import LabelEncoder 

# 그래프 기본 테마 설정
sns.set_theme(palette="tab10", style="darkgrid", font_scale=1)
sns.color_palette("tab10", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Umdot 12'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

# 파~일을 열어다오~
- 아... 두개네... 

In [ ]:
# 파일이 두개더군요... 
artist_df = pd.read_csv(f'{path}/artists.csv') 
song_df = pd.read_csv(f'{path}/songs.csv') 

## .info()

In [ ]:
artist_df.info()

In [ ]:
song_df.info()

## .describe()

In [ ]:
artist_df.describe()

In [ ]:
artist_df.describe(include = 'O')

In [ ]:
song_df.describe()

In [ ]:
song_df.describe(include = 'O')

## .isna()sum()

In [ ]:
artist_df.isna().sum()

In [ ]:
song_df.isna().sum()

## .head()

In [ ]:
artist_df.head()

In [ ]:
song_df.head()

## .columnns

In [ ]:
artist_df.columns

In [ ]:
song_df.columns

# 전처리

## 결측값 확인

In [ ]:
na_artist = artist_df.query('name.isna()').index
# 아이디로 조회가 안되는데 아이디가 왜 있는건지 모르겠음. 
for idx in na_artist:
    artist_df.loc[idx, 'name'] = 'unknown'

artist_df.isna().sum()

In [ ]:
na1_song = song_df.query('name.isna()').index
# 곡명이 언노운이면 뭐 나보고 어쩌라는겨... 

na2_song = song_df.query('album_name.isna()').index
# 앨범이 언노운이면 뭐 우째야되나... 

for idx in na1_song:
    song_df.loc[idx, 'name'] = 'untitled'

for idx in na2_song:
    song_df.loc[idx, 'album_name'] = 'various'

song_df.isna().sum()

## 범주화

### 아티스트

In [ ]:
# 팔로워 범주화 
# 1억 저건 뭐 브루노마스임? ㄷㄷ 
artist_df['followers'].min(), artist_df['followers'].max()

artist_df['follower_group'] = pd.qcut(artist_df['followers'], q=10, labels=range(1, 11))

In [ ]:
popularity_bins = list(range(0, 110, 10))
popularity_label = list(range(0, 100, 10))

# 이걸 넣어주면 0도 포함됩니다. 
artist_df['popularity_group'] = pd.cut(artist_df['popularity'], bins = popularity_bins, labels = popularity_label, include_lowest=True)

In [ ]:
artist_df

### 노래

In [ ]:
song_df['year'].min(), song_df['year'].max()

# 노래들 년도 볌주화
song_era = list(range(1900, 2040, 10)) # 1900년대부터 10틱으로 갑니다. 
song_era_label = [f"{i}s" for i in song_era[:-1]]

song_df['Era'] = pd.cut(song_df['year'], bins = song_era, labels = song_era_label, include_lowest = True, right = False)


In [ ]:
popularity_bins = list(range(0, 110, 10))
popularity_label = list(range(0, 100, 10))

# 이걸 넣어주면 0도 포함됩니다. 
song_df['popularity_group'] = pd.cut(song_df['popularity'], bins = popularity_bins, labels = popularity_label, include_lowest=True)

In [ ]:
song_df['follower_group'] = pd.qcut(song_df['total_artist_followers'], q=10, labels=range(1, 11))

In [ ]:
song_df

- 생각보다 범주화할 건 얼마 없는데 양이 많아서 그런가 리눅스가 다 뻗는다... 여러분들 VScode 뻗는거 보셨습니까? 저는 봤습니다. 

# 분석해보기

## 아티스트

### 팔로워

In [ ]:
artist_df.groupby('follower_group')['id'].count()

In [ ]:
# 1그룹에 다 몰렸구나... 
# 결측값을 0으로 채움
genre_follower = artist_df.groupby(['main_genre','follower_group'], observed=True)['id'].size().unstack()

# 각 장르별 합계
genre_follower['total_group'] = genre_follower.sum(axis=1)

# 정렬
genre_sort = genre_follower.sort_values('total_group', ascending=False)
genre_sort

In [ ]:
sns.barplot(genre_sort, x = 'main_genre', y = 'total_group', hue = 'main_genre')
plt.xlabel('장르')
plt.ylabel('아티스트')
plt.show()

- TOP 3은 일렉트로닉, 락, 팝이다. 
- 1, 2위가 되게 독보적이다. 

### 인기도

In [ ]:
artist_df.groupby('popularity_group')['id'].count()

In [ ]:
# 1그룹에 다 몰렸구나... 
# 결측값을 0으로 채움
genre_popular = artist_df.groupby(['main_genre','popularity_group'], observed=True)['id'].size().unstack().fillna(0)

# 각 장르별 합계
genre_popular['total_group'] = genre_popular.sum(axis=1)

# 정렬
genre_sort = genre_popular.sort_values('total_group', ascending=False)
genre_sort

In [ ]:
sns.barplot(genre_sort, x = 'main_genre', y = 'total_group', hue = 'main_genre')
plt.xlabel('장르')
plt.ylabel('아티스트')
plt.show()

- 장르별로는 팔로워나 인기나 또이또이 쌤샘인 듯 하다. 

In [ ]:
# 장르별, 팔로우 그룹별 시각화
genre_follower_hist = artist_df.groupby(['main_genre','follower_group'], observed=True)['id'].size().unstack()

# 정규화
genre_norm = genre_follower_hist.div(genre_follower_hist.sum(axis=1), axis=0)

# 시각화
genre_norm.plot(kind='barh', stacked=True, figsize=(12, 8), colormap='viridis')

plt.title('장르별 인기도 그룹 비중 (정규화)', fontsize=15)
plt.xlabel('비중 (Percentage)')
plt.ylabel('주요 장르')
plt.legend(title='인기도 그룹', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 장르별, 인기도별 시각화
genre_popular_hist = artist_df.groupby(['main_genre','popularity_group'], observed=True)['id'].size().unstack()

# 정규화
genre_norm = genre_popular_hist.div(genre_popular_hist.sum(axis=1), axis=0)

# 시각화
genre_norm.plot(kind='barh', stacked=True, figsize=(12, 8), colormap='viridis')

plt.title('장르별 인기도 그룹 비중 (정규화)', fontsize=15)
plt.xlabel('비중 (Percentage)')
plt.ylabel('주요 장르')
plt.legend(title='인기도 그룹', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

- 아티스트는 여기까지 하고 노래로 넘어가도 될듯? 
- 팔로워 수가 제일 많았던 일렉트로닉스는 분포가 생각보다 이븐하다. 

### 상위 10% 팔로워 수를 가진 아티스트들의 장르는?

In [ ]:
# 팔로워 상위 10%
top_10_percent = artist_df[artist_df['follower_group'] == 10]

top_10_genre = top_10_percent.groupby('main_genre').size()
top_10_genre_df = top_10_genre_s.reset_index()
top_10_genre_df.columns = [*top_10_genre_df.columns[:-1], 'artist_count']
top_10_genre_s = top_10_genre_df.sort_values('artist_count')

sns.barplot(data=top_10_genre_df, x='main_genre', y='artist_count', hue='main_genre', legend=False)
plt.xlabel('장르')
plt.ylabel('아티스트 수')
plt.show()

In [ ]:
# Pop+인기도 탑텐
bruno_candidates = artist_df[(artist_df['main_genre'] == 'Pop') & (artist_df['popularity'] >= 90)]
display(bruno_candidates[['name', 'popularity']].sort_values(by='popularity', ascending=False))

In [ ]:
# TOP 아티스트 시각화 예시
plt.figure(figsize=(10, 6))
sns.barplot(data=bruno_candidates, x='popularity', y='name', palette='Reds_r')
plt.title('Spotify Popularity TOP Artists', fontsize=16)
plt.xlim(85, 101) # 90점대 근처를 자세히 보기 위해
plt.show()

- 시런형... 의외로 팔로워 많이 없구나... 
- 셀레스티얼 들어보십쇼. 

## 노래

### 년도별로 몇곡이나 있나?

In [ ]:
# 년도가 10년단위라서... 임시로 갑니다. 김람다씨! 
song_df['Era_year'] = song_df['year'].apply(lambda x: f"{(int(x) // 100) * 100}s")

In [ ]:
song_df.groupby(['Era_year','Era'], observed=False).size().unstack()

In [ ]:
plt.figure(figsize=(10, 6))
# 1900s vs 2000s 인기도 분포 비교
sns.boxplot(data=song_df, x='Era_year', y='popularity', hue='Era_year', legend=False)

plt.title('20세기 vs 21세기 곡 인기도 분포 비교', fontsize=16)
plt.show()

In [ ]:
song_df.query('popularity > 80 and Era_year == "1900s"').sort_values('popularity', ascending=False).head()

In [ ]:
song_df.query('popularity > 80 and Era_year == "2000s"').sort_values('popularity', ascending=False).head()

### 각 시대별 명곡 찾기 -1990s

In [ ]:
song_1990 = song_df.query('Era_year == "1900s"')

era_boss_songs = song_1990.loc[song_1990.groupby('Era', observed=True)['popularity'].idxmax()]
display(era_boss_songs[['Era', 'artists', 'name', 'popularity']].sort_values('Era'))

In [ ]:
# 인기도 80 이상만 명확하게 표시
top_classics = song_1990[song_1990['popularity'] >= 80]

plt.figure(figsize=(12, 8))
# KDE Plot (밀도 등고선) 추가
sns.kdeplot(data=song_1990, x='danceability', y='energy', levels=5, color="black", linewidths=1)
# 전체 배경은 연하게
sns.scatterplot(data=song_1990, x='danceability', y='energy', color='lightgrey', alpha=0.1)
# 띵곡들만 진하게
sns.scatterplot(data=top_classics, x='danceability', y='energy', 
                hue='popularity', size='popularity', palette='Purples', sizes=(50, 300))
plt.title('1990년대의 띵곡')
plt.show()

### 각 시대별 명곡 찾기-2000s

In [ ]:
song_2000 = song_df.query('Era_year == "2000s"')

era_boss_songs = song_2000.loc[song_2000.groupby('Era', observed=True)['popularity'].idxmax()]
display(era_boss_songs[['Era', 'artists', 'name', 'popularity']].sort_values('Era'))

In [ ]:
# 인기도 80 이상만 명확하게 표시
top_classics = song_2000[song_2000['popularity'] >= 80]

plt.figure(figsize=(12, 8))
# KDE Plot (밀도 등고선) 추가
sns.kdeplot(data=song_2000, x='danceability', y='energy', levels=5, color="black", linewidths=1)
# 전체 배경은 연하게
sns.scatterplot(data=song_2000, x='danceability', y='energy', color='lightgrey', alpha=0.1)
# 띵곡들만 진하게
sns.scatterplot(data=top_classics, x='danceability', y='energy', 
                hue='popularity', size='popularity', palette='Purples', sizes=(50, 300))
plt.title('2000년대의 띵곡')
plt.show()

In [ ]:
# 1930년 이전의 진짜 조상님 곡들 리스트업
old_legends = song_df[song_df['year'] < 1930].sort_values('popularity', ascending=False)
display(old_legends[['year', 'artists', 'album_name', 'popularity']].head(10))

In [ ]:
# 1950년 이전 출시 곡들 중 '올타임 레전드' 찾기
legend_check = song_df[song_df['year'] < 1950].nlargest(5, 'popularity')
display(legend_check[['year', 'artists', 'album_name', 'popularity']])

In [ ]:
# 'Christmas' 단어가 들어간 곡 제외하고 1940년대 띵곡 찾기
no_carol_40s = song_df[(song_df['year'] < 1950) & 
                        (~song_df['album_name'].str.contains('Christmas|Jingle|Santa', case=False))]
display(no_carol_40s.nlargest(5, 'popularity')[['year', 'artists', 'album_name', 'popularity']])

- 아는 곡이 거의 없음... 

## 아티스트별 띵곡
### 마이클 잭슨
- 나도 마잭은 알아요. 

In [ ]:
# 마이클 잭슨
song_df[song_df['artists'].str.contains('Michael Jackson', case=False)]

In [ ]:
# 구조된 마이클 잭슨 곡의 '민낯' 공개
mj_the_one = song_df[song_df['artists'].str.contains('Michael Jackson', case=False, na=False)].sort_values('popularity', ascending=False)   

# 데이터 수치 확인 (NanumSquare 폰트로 출력!)
display(mj_the_one[['name', 'year', 'popularity', 'danceability', 'energy', 'valence']][0:5])

### 에드시런
- 셀레스티얼! (SV 엔딩 크레딧곡)

In [ ]:
# 에드시런 찾아 삼만리
mj_the_one = song_df[song_df['artists'].str.contains('Ed Sheeran', case=False, na=False)].sort_values('popularity', ascending=False)   

# 데이터 수치 확인 (NanumSquare 폰트로 출력!)
display(mj_the_one[['name', 'year', 'popularity', 'danceability', 'energy', 'valence']][0:5])

#

### ~~월클~~ BTS
- 저는 아미는 아니고요... 있나 해서 쳐봤더니 있더라고... 

In [ ]:
# 방탄 월클 맞다
mj_the_one = song_df[song_df['artists'].str.contains('BTS', case=False, na=False)].sort_values('popularity', ascending=False)   

# 데이터 수치 확인 (NanumSquare 폰트로 출력!)
display(mj_the_one[['name', 'year', 'popularity', 'danceability', 'energy', 'valence']][0:5])